# Voice Intelligence — Review Analysis Pipeline

Analiza reviews de Reddit sobre Holafly usando Claude para extraer insights creativos.

1. Configuracion
2. Carga de datos desde CSV
3. Limpieza y filtrado
4. Analisis con Claude (async, batched)
5. Resultados y exportacion
6. Generacion de briefs creativos

## 1. Configuracion

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../../.env")

ANTHROPIC_API_KEY = os.environ["CLAUDE-API_KEY"]

# Ruta al CSV
CSV_PATH = "data/app store US holafly - dataset_appstore-reviews-scraper_2026-04-23_18-08-17-782.csv"

# Nombre EXACTO de la columna con el texto de la review
REVIEW_TEXT_COLUMN = "content"

# Brand config
BRAND_CONFIG = {
    "name": "Holafly",
    "category": "eSIM internacional para viajeros (datos moviles sin roaming)",
    "audience": (
        "Viajeros internacionales, nomades digitales, turistas "
        "que buscan conectividad sin depender de SIM fisicas o roaming"
    ),
    "nonnegotiables": [
        "No mencionar competidores por nombre (Airalo, Nomad, SimOptions)",
        "Tono viajero, accesible, no telecom corporativo",
        "Resaltar la simplicidad de activacion y la tranquilidad de estar conectado",
    ],
}

# LLM params
MODEL = "claude-sonnet-4-6"
BATCH_SIZE = 10       # reviews por llamada al LLM
MAX_CONCURRENT = 8    # llamadas paralelas
MAX_REVIEWS = None    # None = todas, o un int para limitar

## 2. Carga de datos desde CSV

In [ ]:
import pandas as pd

df_raw = pd.read_csv(CSV_PATH)
print(f"Cargadas {len(df_raw)} filas")
print(f"\nColumnas disponibles: {df_raw.columns.tolist()}")
df_raw.head(3)

## 3. Limpieza y filtrado

In [ ]:
assert REVIEW_TEXT_COLUMN in df_raw.columns, (
    f"La columna '{REVIEW_TEXT_COLUMN}' no existe. "
    f"Elegi una de: {df_raw.columns.tolist()}"
)

df = df_raw.copy()
df = df.dropna(subset=[REVIEW_TEXT_COLUMN])
df[REVIEW_TEXT_COLUMN] = df[REVIEW_TEXT_COLUMN].astype(str)
df = df.drop_duplicates(subset=[REVIEW_TEXT_COLUMN])
df = df[df[REVIEW_TEXT_COLUMN].str.len() > 30].reset_index(drop=True)

if MAX_REVIEWS:
    df = df.head(MAX_REVIEWS)

reviews = df[REVIEW_TEXT_COLUMN].tolist()
print(
    f"{len(reviews)} reviews listas para analizar "
    f"(despues de limpiar duplicados y basura)"
)

## 4. Prompt de analisis + motor async

In [ ]:
import asyncio
import json

import nest_asyncio
from anthropic import AsyncAnthropic

nest_asyncio.apply()

client_async = AsyncAnthropic(api_key=ANTHROPIC_API_KEY)

ANALYSIS_PROMPT = """You are a consumer insights analyst for Winclap, a growth marketing agency. \
You analyze real consumer voice data to extract creative advertising insights.

BRAND CONTEXT:
- Brand: {brand_name}
- Category: {category}
- Target audience: {audience}

Your task: analyze a batch of user reviews and extract structured insights.

Return ONLY a valid JSON array with one object per review, in the same order as the input. \
For each review provide:

{{
  "theme": a 2-3 word theme tag IN ENGLISH, lowercase_with_underscores \
(e.g., "shipping_speed", "vendor_reliability", "app_ux", "price_value", \
"customer_support", "trust_refund", "coverage_availability", "activation_setup", \
"product_quality", "search_discovery"),
  "sentiment": "positive" | "negative" | "neutral" | "mixed",
  "emotion": one single dominant emotion as one word (relief, frustration, delight, \
anxiety, surprise, gratitude, regret, pride, disappointment, resignation, etc.),
  "funnel_stage": "awareness" | "consideration" | "purchase" | "activation" | \
"usage" | "retention",
  "job_to_be_done": one specific sentence -- what was the user trying to accomplish?,
  "pain_or_delight": the concrete friction or aha moment in their own words (NOT abstract),
  "moment": the physical/emotional situation in one short phrase \
(e.g., "gift arrival on child's birthday", "first-time activation at foreign airport"),
  "quotable": true if this review contains a vivid verbatim phrase usable in an ad, else false,
  "verbatim_quote": exact snippet max 25 words if quotable, else null,
  "creative_potential": integer 1-10 on how useful this review is as creative inspiration,
  "strategic_signal": if this review reveals a product/positioning opportunity, \
describe it in one sentence; else null
}}

Reviews to analyze (numbered, keep order in output):
{reviews_block}

Return ONLY the JSON array. No preamble, no markdown fences, no explanation."""

In [ ]:
def _strip_fences(text: str) -> str:
    """Remove markdown code fences if Claude adds them."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    return text


async def analyze_batch(
    reviews_batch: list[str],
    semaphore: asyncio.Semaphore,
    batch_idx: int,
) -> list[dict | None]:
    """Analyze a single batch of reviews via Claude."""
    async with semaphore:
        reviews_block = "\n\n".join(
            f"[{i+1}] {r[:900]}" for i, r in enumerate(reviews_batch)
        )
        prompt = ANALYSIS_PROMPT.format(
            brand_name=BRAND_CONFIG["name"],
            category=BRAND_CONFIG["category"],
            audience=BRAND_CONFIG["audience"],
            reviews_block=reviews_block,
        )
        try:
            resp = await client_async.messages.create(
                model=MODEL,
                max_tokens=4500,
                messages=[{"role": "user", "content": prompt}],
            )
            return json.loads(_strip_fences(resp.content[0].text))
        except Exception as e:
            print(f"  batch {batch_idx} error: {e}")
            return [None] * len(reviews_batch)


async def analyze_all(reviews: list[str]) -> list[dict | None]:
    """Run all batches concurrently with a semaphore cap."""
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    batches = [
        reviews[i : i + BATCH_SIZE]
        for i in range(0, len(reviews), BATCH_SIZE)
    ]
    print(
        f"Analizando {len(reviews)} reviews en "
        f"{len(batches)} batches de {BATCH_SIZE}..."
    )
    tasks = [
        analyze_batch(b, semaphore, i) for i, b in enumerate(batches)
    ]
    results_nested = await asyncio.gather(*tasks)
    results = []
    for batch_result in results_nested:
        results.extend(batch_result)
    return results

## 5. Correr analisis y exportar resultados

In [ ]:
import time

start = time.time()
analyses = asyncio.run(analyze_all(reviews))

df_analysis = pd.DataFrame([a if a else {} for a in analyses])
df_full = pd.concat([df.reset_index(drop=True), df_analysis], axis=1)

elapsed = time.time() - start
print(f"\nAnalisis completo en {elapsed:.1f} seg")
print(
    f"{df_full['theme'].notna().sum()} / {len(df_full)} "
    f"reviews clasificadas exitosamente"
)
print(f"\nDistribucion por tema:")
df_full["theme"].value_counts().head(10)

In [ ]:
OUTPUT_CSV = "output/reviews_analyzed.csv"
OUTPUT_JSON = "output/analyses.json"

df_full.to_csv(OUTPUT_CSV, index=False)
df_full.to_json(OUTPUT_JSON, orient="records", force_ascii=False, indent=2)

print(f"Guardado: {OUTPUT_CSV} y {OUTPUT_JSON}")

## 6. (Opcional) Generar 10 briefs creativos

In [ ]:
from anthropic import Anthropic

client_sync = Anthropic(api_key=ANTHROPIC_API_KEY)

BRIEFS_PROMPT = """You are the Creative Strategy Director at Winclap. \
You just received a Voice Intelligence Report from {brand_name}'s customers. \
Your job: produce 10 creative briefs for paid social ads (Meta, TikTok, YouTube) \
that would resonate because they are grounded in REAL consumer voice.

THE RULE: every brief must be traceable to a specific customer quote. \
No generic marketing speak.

BRAND CONTEXT:
- Brand: {brand_name}
- Category: {category}
- Audience: {audience}
- Non-negotiables: {nonnegotiables}

INPUT DATA -- clustered customer signals:
{cluster_summaries}

TOP VERBATIM QUOTES BY THEME:
{top_quotes}

Generate 10 creative briefs. Must be DIVERSE across:
- angle (emotional / rational / social_proof / fomo / problem_agitation / transformation)
- format (UGC_video / static / carousel / reaction_video / testimonial / founder_pov / comparison)
- funnel stage (awareness / consideration / conversion)
- customer segment (new / repeat / frustrated switcher / skeptic / power user)

For each brief return JSON:
{{
  "brief_id": 1-10,
  "title": short memorable name,
  "hook": verbatim or tight paraphrase from a real customer (the first 3 seconds of the ad),
  "insight": the underlying human truth (one sentence),
  "customer_quote_source": the literal quote from the input data that inspired this,
  "angle": one of the 6 angles,
  "format": one of the 7 formats,
  "funnel_stage": awareness | consideration | conversion,
  "target_segment": who this ad is for,
  "script_concept": 2-3 sentences describing what happens,
  "cta": specific call to action,
  "why_this_wins": one sentence on why this beats typical category ads
}}

Output a valid JSON array of exactly 10 briefs. No text before or after."""

In [ ]:
def generate_briefs(df_full: pd.DataFrame) -> tuple[list[dict], list[dict]]:
    """Generate creative briefs from analyzed review data."""
    df_valid = df_full.dropna(subset=["theme"])

    # Cluster by theme
    clusters = []
    for theme, group in df_valid.groupby("theme"):
        top_quotes = (
            group[group["quotable"] == True]
            .sort_values("creative_potential", ascending=False)["verbatim_quote"]
            .dropna()
            .head(5)
            .tolist()
        )
        clusters.append({
            "theme": theme,
            "volume": len(group),
            "pct_of_total": round(len(group) / len(df_valid) * 100, 1),
            "top_emotions": group["emotion"].value_counts().head(3).to_dict(),
            "top_moments": (
                group["moment"].dropna().value_counts().head(5).index.tolist()
            ),
            "top_jobs": (
                group["job_to_be_done"]
                .dropna()
                .value_counts()
                .head(3)
                .index.tolist()
            ),
            "top_quotes": top_quotes,
        })
    clusters.sort(key=lambda c: c["volume"], reverse=True)

    # Top 30 quotes overall
    top_quotes_global = (
        df_valid[df_valid["quotable"] == True]
        .sort_values("creative_potential", ascending=False)["verbatim_quote"]
        .dropna()
        .head(30)
        .tolist()
    )
    top_quotes_text = "\n".join(f'- "{q}"' for q in top_quotes_global)

    cluster_summaries = json.dumps(
        [
            {k: v for k, v in c.items() if k != "top_quotes"}
            for c in clusters[:8]
        ],
        indent=2,
        ensure_ascii=False,
    )

    prompt = BRIEFS_PROMPT.format(
        brand_name=BRAND_CONFIG["name"],
        category=BRAND_CONFIG["category"],
        audience=BRAND_CONFIG["audience"],
        nonnegotiables="; ".join(BRAND_CONFIG["nonnegotiables"]),
        cluster_summaries=cluster_summaries,
        top_quotes=top_quotes_text,
    )

    resp = client_sync.messages.create(
        model=MODEL,
        max_tokens=8000,
        messages=[{"role": "user", "content": prompt}],
    )
    briefs = json.loads(_strip_fences(resp.content[0].text))
    return briefs, clusters


briefs, clusters = generate_briefs(df_full)

In [ ]:
# Guardar reporte final
final_report = {
    "meta": {
        "brand": BRAND_CONFIG["name"],
        "total_reviews": len(df_full),
        "runtime_seconds": round(elapsed, 1),
    },
    "clusters": clusters,
    "briefs": briefs,
}

with open("output/voice_report.json", "w") as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False)

print(f"Generados {len(briefs)} briefs creativos")
print(f"Guardado: output/voice_report.json")
print("\n--- PRIMER BRIEF DE EJEMPLO ---")
print(json.dumps(briefs[0], indent=2, ensure_ascii=False))